In [1]:
import cv2
import torch
import torchvision

In [2]:
from torch2trt import TRTModule

model_trt = TRTModule()
model_trt.load_state_dict(torch.load('road_following_model_trt.pth'))

collision_model_trt = TRTModule()
collision_model_trt.load_state_dict(torch.load('best_model_trt.pth'))

avoidance_model_trt = TRTModule()
avoidance_model_trt.load_state_dict(torch.load('car_model_trt.pth'))

<All keys matched successfully>

In [3]:
from jetracer.nvidia_racecar import NvidiaRacecar

car = NvidiaRacecar()

In [4]:
#import traitlets
from jetcam.csi_camera import CSICamera
#import ipywidgets
#from IPython.display import display
#from jetcam.utils import bgr8_to_jpeg
#from jupyter_clickable_image_widget import ClickableImageWidget


camera = CSICamera(width=224, height=224, capture_fps=24)


# create image preview
#camera.running = True
#camera_widget = ClickableImageWidget(width=camera.width, height=camera.height)
#traitlets.dlink((camera, 'value'), (camera_widget, 'value'), transform=bgr8_to_jpeg)
#display(camera_widget)

In [ ]:
import torch
import torch.nn.functional as F
from utils import preprocess
import numpy
import threading
import time

# Road following variables
STEERING_GAIN = 0.85
STEERING_BIAS = -0.05
THROTTLE_SPEED = 0.15

# Collision variables
STOPPED_THRESHOLD = 0.13
STOP_DURATION = 3.0
last_stopped_time = 0
stopped = False
prob_stopped = 0.0
stop_sign_active = False 



# Avoidance variables
STEERING_BIAS_RIGHT = THROTTLE_SPEED * 3.0
STEERING_BIAS_RIGHT_DURATION = 0.35
STEERING_BIAS_LEFT = THROTTLE_SPEED * 3.5
STEERING_BIAS_LEFT_DURATION = 0.7
BLOCKED_THRESHOLD = 0.5
AVOIDANCE_COOLDOWN = 3.0
switching = False
last_avoidance_check = 0
prob_blocked = 0.0
steering_time = 0
steering = True



# Threading
shared_frame = None
lock = threading.Lock()

def frame_capture_thread_func():
    global shared_frame
    while True:
        frame = camera.read()
        with lock:
            shared_frame = frame
        time.sleep(0.01)

# Collision thread
def collision_thread_func():
    global prob_stopped
    while True:
        with lock:
            if shared_frame is None:
                continue
            image = shared_frame.copy()
        collision_input_image = preprocess(image).float()
        with torch.no_grad():
            output = collision_model_trt(collision_input_image)
            probs = F.softmax(output, dim=1)
            prob_stopped_local = float(probs[0, 0])
        with lock:
            prob_stopped = prob_stopped_local
        time.sleep(0.05)
        
# Avoidance thread
def avoidance_thread_func():
    global prob_blocked
    while True:
        with lock:
            if shared_frame is None:
                continue
            image = shared_frame.copy()
        avoidance_input_image = preprocess(image).float()
        with torch.no_grad():
            output = avoidance_model_trt(avoidance_input_image)
            probs = F.softmax(output, dim=1)
            prob_blocked_local = float(probs[0, 1])
        with lock:
            prob_blocked = prob_blocked_local
        time.sleep(0.05)
                
# Start threads
threading.Thread(target=frame_capture_thread_func, daemon=True).start()
threading.Thread(target=collision_thread_func, daemon=True).start()
threading.Thread(target=avoidance_thread_func, daemon=True).start()

# Main loop
while True:
    with lock:
        if shared_frame is None:
            continue
        image = shared_frame.copy()
    input_image = preprocess(image).half()
    
    # Steering inference
    with torch.no_grad():
        output = model_trt(input_image).cpu().numpy().flatten()
        x = float(output[0])
    
    current_time = time.time()
    
    # Steering logic
    with lock:
        is_blocked = prob_blocked >= BLOCKED_THRESHOLD
    
    # Detect obstacle
    if (is_blocked and not switching and 
        current_time - last_avoidance_check > AVOIDANCE_COOLDOWN):
        switching = True
        steering = True
        steering_time = current_time
        last_avoidance_check = current_time
        print(f"(!) OBSTACLE detected (P = {prob_blocked:.2f}) — SWITCHING LANES.")
    
    # When switching lanes, switch to the right lane and back to the left lane within LANE_SWITCH_DURATION
    if switching:
        steering_elapsed = current_time - steering_time
        if steering:  # Turn right
            car.steering = x * STEERING_GAIN + STEERING_BIAS + STEERING_BIAS_RIGHT
            if steering_elapsed > STEERING_BIAS_RIGHT_DURATION:
                steering = False
                steering_time = current_time
        else:  # Turn left
            car.steering = x * STEERING_GAIN + STEERING_BIAS - STEERING_BIAS_LEFT
            if current_time - steering_time > STEERING_BIAS_LEFT_DURATION:
                switching = False
    else:
        car.steering = x * STEERING_GAIN + STEERING_BIAS

        
    if not switching:
        car.steering = x * STEERING_GAIN + STEERING_BIAS
    
    # Throttle logic
    with lock:
        is_stopped = prob_stopped >= STOPPED_THRESHOLD

    # Detect stop sign
    if is_stopped and not stopped and not stop_sign_active:
        stopped = True
        stop_sign_active = True
        last_stopped_time = current_time
        print(f"(X) STOP SIGN detected (P = {prob_stopped:.2f}) — STOPPING for 3 seconds.")
    
    # When stopped, wait for STOP_DURATION before continuing
    if stopped:
        if current_time - last_stopped_time < STOP_DURATION:
            car.throttle = 0
        else:
            stopped = False
            car.throttle = THROTTLE_SPEED
    else:
        car.throttle = THROTTLE_SPEED

    # Reset stop_sign_active when clear
    if not stopped and not is_stopped:
        stop_sign_active = False

(!) OBSTACLE detected (P = 0.55) — SWITCHING LANES.
(!) OBSTACLE detected (P = 0.57) — SWITCHING LANES.
(X) STOP SIGN detected (P = 0.40) — STOPPING for 3 seconds.
(!) OBSTACLE detected (P = 0.59) — SWITCHING LANES.
(!) OBSTACLE detected (P = 0.57) — SWITCHING LANES.
(!) OBSTACLE detected (P = 0.62) — SWITCHING LANES.
(X) STOP SIGN detected (P = 0.20) — STOPPING for 3 seconds.
(!) OBSTACLE detected (P = 0.53) — SWITCHING LANES.
(!) OBSTACLE detected (P = 0.56) — SWITCHING LANES.
(X) STOP SIGN detected (P = 0.21) — STOPPING for 3 seconds.
(!) OBSTACLE detected (P = 0.56) — SWITCHING LANES.
(!) OBSTACLE detected (P = 0.54) — SWITCHING LANES.
(X) STOP SIGN detected (P = 0.58) — STOPPING for 3 seconds.
(!) OBSTACLE detected (P = 0.71) — SWITCHING LANES.
